### Partie 1 – Exploration du dataset 

In [3]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd

In [4]:
RAW_DIR = Path("../data/raw")
print(RAW_DIR)
print(RAW_DIR.exists())

..\data\raw
True


In [5]:
classes = [d.name for d in RAW_DIR.iterdir() if d.is_dir()]

print(classes)

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [14]:
resultats = []

for classe in classes:
    dossier_classe = RAW_DIR / classe

    for fichier in sorted(dossier_classe.iterdir()):
        if not fichier.is_file():
            continue

        # Infos de base (valables même si l'image est corrompue)
        info = {
            "nom": fichier.name,
            "classe": classe,
            "format": None,
            "mode": None,
            "largeur": None,
            "hauteur": None,
            "ecart_type": None,
            "canaux": None,
            "taille": fichier.stat().st_size,  # en octets
            "corrompue": False,
            "erreur": None,
        }

        try:
            # 1) Vérification de l'intégrité (verify() rend l'objet inutilisable ensuite)
            with Image.open(fichier) as img:
                img.verify()

            # 2) Réouverture pour lire les attributs
            with Image.open(fichier) as img:
                pixels = np.array(img)  # charge tous les pixels (détecte aussi les fichiers tronqués)
                info["format"] = img.format
                info["mode"] = img.mode
                info["largeur"], info["hauteur"] = img.size
                info["canaux"] = len(img.getbands())
                info["ecart_type"] = float(np.std(pixels))

        except Exception as e:
            info["corrompue"] = True
            info["erreur"] = str(e)

        resultats.append(info)

df = pd.DataFrame(resultats)
print(df.shape)
df

(1032, 11)


,nom,classe,format,mode,largeur,hauteur,ecart_type,canaux,taille,corrompue,erreur
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False,NaN
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False,NaN
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False,NaN
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False,NaN
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1027,trash50.jpg,trash,JPEG,RGB,512.0,384.0,41.780027,3.0,13563,False,NaN
1028,trash6.jpg,trash,JPEG,RGB,512.0,384.0,37.391702,3.0,10534,False,NaN
1029,trash7.jpg,trash,JPEG,RGB,512.0,384.0,41.297376,3.0,8011,False,NaN
1030,trash8.jpg,trash,JPEG,RGB,512.0,384.0,47.635953,3.0,16946,False,NaN


### Partie 2 – Détecter les images corrompues 

##### Écrire et se servir d’une fonction qui détecte une image corrompue. 

In [15]:
def est_corrompue(chemin):
    try:
        
        with Image.open(chemin) as img:
            img.verify()

        with Image.open(chemin) as img:
            img.load()

        return False, None

    except Exception as e:
        return True, str(e)

In [17]:
corrompues = []   # liste des images corrompues
total = 0         # nombre total d'images analysées

for classe in classes:
    for fichier in (RAW_DIR / classe).iterdir():
        if fichier.is_file():
            total += 1
            corrompue, erreur = est_corrompue(fichier)

            if corrompue:
                corrompues.append({
                    "nom": fichier.name,
                    "classe": classe,
                    "erreur": erreur,
                })

print("Total d'images analysées :", total)
print("Images corrompues        :", len(corrompues))

for c in corrompues:
    print(c)

Total d'images analysées : 1032
Images corrompues        : 6
{'nom': 'cardboard83.jpg', 'classe': 'cardboard', 'erreur': "cannot identify image file '..\\\\data\\\\raw\\\\cardboard\\\\cardboard83.jpg'"}
{'nom': 'glass74.jpg', 'classe': 'glass', 'erreur': 'Truncated File Read'}
{'nom': 'metal48.jpg', 'classe': 'metal', 'erreur': "cannot identify image file '..\\\\data\\\\raw\\\\metal\\\\metal48.jpg'"}
{'nom': 'paper213.jpg', 'classe': 'paper', 'erreur': "cannot identify image file '..\\\\data\\\\raw\\\\paper\\\\paper213.jpg'"}
{'nom': 'plastic13.jpg', 'classe': 'plastic', 'erreur': "cannot identify image file '..\\\\data\\\\raw\\\\plastic\\\\plastic13.jpg'"}
{'nom': 'trash3.jpg', 'classe': 'trash', 'erreur': "cannot identify image file '..\\\\data\\\\raw\\\\trash\\\\trash3.jpg'"}


### Partie 3 – Détecter les images vides 
##### Écrire et se servir d’une fonction qui détecte les images vides : image entièrement noire, image entièrement blanche ou image dont les pixels présentent très peu de variation. 

In [18]:
def est_vide(chemin, seuil_noir=10, seuil_blanc=245, seuil_std=5):
    with Image.open(chemin) as img:
        gris = img.convert("L")            # ramène RGB / RGBA / grayscale à 1 canal (0-255)
        pixels = np.array(gris)

    moyenne = pixels.mean()
    ecart_type = pixels.std()

    # Peu de variation : les pixels sont presque tous identiques
    if ecart_type < seuil_std:
        if moyenne < seuil_noir:
            return True, "image noire"
        elif moyenne > seuil_blanc:
            return True, "image blanche"
        else:
            return True, "peu de variation"

    return False, None

In [19]:
resultats_vides = []

for classe in classes:
    for fichier in (RAW_DIR / classe).iterdir():
        if not fichier.is_file():
            continue

        # Une image corrompue ne peut pas être analysée : on l'ignore ici
        corrompue, _ = est_corrompue(fichier)
        if corrompue:
            continue

        vide, raison = est_vide(fichier)
        resultats_vides.append({
            "nom": fichier.name,
            "classe": classe,
            "vide": vide,
            "raison_vide": raison,
        })

df_vides = pd.DataFrame(resultats_vides)

nb_vides = int(df_vides["vide"].sum())
print("Images quasi vides :", nb_vides)
print(df_vides[df_vides["vide"]]["raison_vide"].value_counts())

df_vides[df_vides["vide"]]   # liste des images concernées

Images quasi vides : 4
raison_vide
image blanche    2
image noire      2
Name: count, dtype: int64


,nom,classe,vide,raison_vide
166,image-blanche-512x384.jpg,cardboard,True,image blanche
353,image-noire-512x384.png,glass,True,image noire
355,image-blanche-512x384.jpg,metal,True,image blanche
356,image-noire-512x384.png,metal,True,image noire
